In [0]:
# Communication Agent — sends Teams alerts for escalated exceptions
# Uses Microsoft Teams Incoming Webhook (simpler than MAF for this use case)
# In production: use Microsoft Graph API for full Teams integration

from pyspark.sql import functions as F
import requests
import json
from datetime import datetime
import mlflow

CATALOG       = "p2p_databricks"

# ── Teams webhook URL ──────────────────────────────────────────────────────
# Get this from Teams: channel → ... → Connectors → Incoming Webhook
# For portfolio demo, set to None to run in DRY_RUN mode
TEAMS_WEBHOOK = None   # replace with your webhook URL when ready
DRY_RUN       = TEAMS_WEBHOOK is None

print(f"Communication Agent ready")
print(f"Mode: {'DRY RUN (no messages sent)' if DRY_RUN else 'LIVE'}")
print(f"Catalog: {CATALOG}")

In [0]:
# Load all escalated exceptions from agent verdicts
escalations = (
    spark.table(f"{CATALOG}.gold.agent_exception_verdicts")
         .filter("agent_verdict = 'ESCALATE'")
         .join(
             spark.table(f"{CATALOG}.gold.reconciliation_results")
                  .select("invoice_number", "invoice_amount",
                          "po_total_amount", "amount_deviation_pct",
                          "invoice_vendor", "po_vendor_name",
                          "gr_date", "invoice_date"),
             on="invoice_number", how="left"
         )
         .select(
             "invoice_number", "reason_code",
             "agent_resolution", "agent_confidence",
             "recommended_action",
             "invoice_amount", "po_total_amount",
             "amount_deviation_pct", "invoice_vendor",
             "po_vendor_name", "gr_date", "invoice_date"
         )
         .collect()
)

print(f"Escalations to notify: {len(escalations)}")
print("\nBreakdown by reason code:")
from collections import Counter
counts = Counter(e["reason_code"] for e in escalations)
for reason, count in sorted(counts.items()):
    print(f"  {reason:25s}: {count}")

In [0]:
def get_priority(reason_code: str) -> tuple:
    """Return (priority_label, color) for each reason code."""
    priority_map = {
        "EXC_DATE":         ("🔴 CRITICAL — Fraud Signal",  "FF0000"),
        "EXC_DUPLICATE":    ("🔴 HIGH — Duplicate Invoice", "FF4500"),
        "EXC_WRONG_VENDOR": ("🟠 HIGH — Vendor Mismatch",   "FF8C00"),
        "EXC_VENDOR":       ("🟠 HIGH — Vendor Mismatch",   "FF8C00"),
        "EXC_PRICE":        ("🟡 MEDIUM — Price Deviation",  "FFA500"),
        "EXC_QTY_MISMATCH": ("🟡 MEDIUM — Qty Mismatch",    "FFA500"),
    }
    return priority_map.get(reason_code,
                             ("⚪ UNKNOWN", "808080"))

def get_routing(reason_code: str) -> str:
    """Which team/person should review this exception."""
    routing = {
        "EXC_DATE":         "Finance Controller + Legal Team",
        "EXC_DUPLICATE":    "AP Manager + Finance Controller",
        "EXC_WRONG_VENDOR": "Procurement Team + AP Manager",
        "EXC_VENDOR":       "Procurement Team + AP Manager",
        "EXC_PRICE":        "AP Manager",
        "EXC_QTY_MISMATCH": "AP Manager + Warehouse Team",
    }
    return routing.get(reason_code, "AP Manager")

def build_teams_message(exc: dict) -> dict:
    """
    Build a structured Teams Adaptive Card message.
    Format: rich card with invoice details, findings, action required.
    """
    priority_label, color = get_priority(exc["reason_code"])
    routing    = get_routing(exc["reason_code"])
    inv_amount = float(exc["invoice_amount"] or 0)
    po_amount  = float(exc["po_total_amount"] or 0)
    deviation  = float(exc["amount_deviation_pct"] or 0)

    # Teams Adaptive Card format
    card = {
        "type": "message",
        "attachments": [{
            "contentType": "application/vnd.microsoft.card.adaptive",
            "content": {
                "$schema": "http://adaptivecards.io/schemas/adaptive-card.json",
                "type":    "AdaptiveCard",
                "version": "1.4",
                "body": [
                    {
                        "type":   "TextBlock",
                        "text":   f"⚠️ Invoice Exception — Action Required",
                        "weight": "Bolder",
                        "size":   "Large",
                        "color":  "Warning"
                    },
                    {
                        "type":  "TextBlock",
                        "text":  priority_label,
                        "weight":"Bolder",
                        "wrap":  True
                    },
                    {
                        "type":   "FactSet",
                        "facts":  [
                            {"title": "Invoice",
                             "value": exc["invoice_number"]},
                            {"title": "Reason",
                             "value": exc["reason_code"]},
                            {"title": "Invoice Amount",
                             "value": f"AED {inv_amount:,.2f}"},
                            {"title": "PO Amount",
                             "value": f"AED {po_amount:,.2f}"},
                            {"title": "Deviation",
                             "value": f"{deviation:+.1f}%"},
                            {"title": "Invoice Vendor",
                             "value": exc["invoice_vendor"] or "N/A"},
                            {"title": "PO Vendor",
                             "value": exc["po_vendor_name"] or "N/A"},
                        ]
                    },
                    {
                        "type":  "TextBlock",
                        "text":  f"**Agent Finding:** {exc['agent_resolution']}",
                        "wrap":  True
                    },
                    {
                        "type":  "TextBlock",
                        "text":  f"**Action Required:** {exc['recommended_action']}",
                        "wrap":  True,
                        "color": "Attention"
                    },
                    {
                        "type":  "TextBlock",
                        "text":  f"**Route to:** {routing}",
                        "wrap":  True
                    },
                    {
                        "type":  "TextBlock",
                        "text":  (f"Agent Confidence: "
                                  f"{exc['agent_confidence']*100:.0f}% | "
                                  f"Processed: {datetime.now().strftime('%Y-%m-%d %H:%M')}"),
                        "size":  "Small",
                        "color": "Default",
                        "isSubtle": True
                    }
                ]
            }
        }]
    }
    return card

print("✅ Teams message builder ready")

# Preview one message
sample = escalations[0]
sample_msg = build_teams_message(sample)
print(f"\nSample message for {sample['invoice_number']}:")
print(json.dumps(
    sample_msg["attachments"][0]["content"]["body"],
    indent=2
)[:800] + "...")

In [0]:
import time

comm_results = []

print(f"Processing {len(escalations)} escalations...")
print(f"Mode: {'DRY RUN' if DRY_RUN else 'LIVE TEAMS'}\n")
print("─" * 65)

# Group by priority — send CRITICAL first
priority_order = [
    "EXC_DATE", "EXC_DUPLICATE",
    "EXC_VENDOR", "EXC_WRONG_VENDOR",
    "EXC_PRICE", "EXC_QTY_MISMATCH"
]

sorted_escalations = sorted(
    escalations,
    key=lambda x: priority_order.index(x["reason_code"])
                  if x["reason_code"] in priority_order else 99
)

for exc in sorted_escalations:
    priority_label, _ = get_priority(exc["reason_code"])
    routing           = get_routing(exc["reason_code"])
    message           = build_teams_message(exc)

    sent    = False
    error   = None

    if DRY_RUN:
        sent = True  # simulate success in dry run
    else:
        try:
            response = requests.post(
                TEAMS_WEBHOOK,
                json    = message,
                timeout = 10
            )
            sent  = response.status_code == 200
            error = response.text if not sent else None
        except Exception as e:
            error = str(e)

    comm_results.append({
        "invoice_number":  exc["invoice_number"],
        "reason_code":     exc["reason_code"],
        "priority":        priority_label,
        "routed_to":       routing,
        "message_sent":    sent,
        "sent_at":         datetime.now().isoformat(),
        "channel":         "Microsoft Teams",
        "dry_run":         DRY_RUN,
        "error":           error or "",
    })

    icon = "📤" if sent else "❌"
    print(f"  {icon} {exc['invoice_number']:25s} | "
          f"{exc['reason_code']:20s} | "
          f"→ {routing}")

    time.sleep(0.2)

print("\n" + "─" * 65)
sent_count = sum(1 for r in comm_results if r["message_sent"])
print(f"  Messages sent    : {sent_count}/{len(comm_results)}")
print(f"  Mode             : {'DRY RUN — no real messages' if DRY_RUN else 'LIVE'}")

In [0]:
from pyspark.sql import types as T

schema = T.StructType([
    T.StructField("invoice_number", T.StringType()),
    T.StructField("reason_code",    T.StringType()),
    T.StructField("priority",       T.StringType()),
    T.StructField("routed_to",      T.StringType()),
    T.StructField("message_sent",   T.BooleanType()),
    T.StructField("sent_at",        T.StringType()),
    T.StructField("channel",        T.StringType()),
    T.StructField("dry_run",        T.BooleanType()),
    T.StructField("error",          T.StringType()),
])

(spark.createDataFrame(comm_results, schema=schema)
      .write.format("delta")
      .mode("overwrite")
      .saveAsTable(f"{CATALOG}.gold.agent_communication_log"))

print(f"✅ Communication log written to {CATALOG}.gold.agent_communication_log")

# Show the log
display(
    spark.table(f"{CATALOG}.gold.agent_communication_log")
         .select("invoice_number", "reason_code",
                 "priority", "routed_to",
                 "message_sent", "sent_at")
         .orderBy("reason_code")
)

In [0]:
mlflow.set_experiment(
    "/Users/shubhammudliar27@outlook.com/p2p_communication_agent"
)

with mlflow.start_run(
    run_name=f"comm_agent_{datetime.now().strftime('%Y%m%d_%H%M')}"
) as run:

    mlflow.log_param("channel",        "Microsoft Teams")
    mlflow.log_param("dry_run",        DRY_RUN)
    mlflow.log_param("routing_rules",  "5 reason codes → 3 teams")
    mlflow.log_param("card_format",    "Teams Adaptive Card v1.4")

    mlflow.log_metric("total_escalations",  len(comm_results))
    mlflow.log_metric("messages_sent",      sent_count)
    mlflow.log_metric("send_success_rate",
                      round(sent_count/len(comm_results), 4))

    # Per priority
    for reason in priority_order:
        subset = [r for r in comm_results
                  if r["reason_code"] == reason]
        if subset:
            mlflow.log_metric(
                f"count_{reason.lower()}",
                len(subset)
            )

    run_id = run.info.run_id

print(f"""
✅ Communication Agent logged to MLflow
{'─'*50}
  Run ID          : {run_id}
  Total escalations: {len(comm_results)}
  Messages sent   : {sent_count}
  Mode            : {'DRY RUN' if DRY_RUN else 'LIVE'}
  Routing rules   :
    EXC_DATE      → Finance Controller + Legal
    EXC_DUPLICATE → AP Manager + Finance Controller
    EXC_VENDOR    → Procurement Team + AP Manager
    EXC_PRICE     → AP Manager
    EXC_QTY       → AP Manager + Warehouse Team
{'─'*50}
""")